# Shuffle e partições

## Task 2: encontrar o shuffle e ajustar o número de partições

In [ ]:
# a sessão Spark deste notebook: já vem pronta (é o código do vídeo 1.4, com outro appName)
import sys
sys.path.insert(0, "../../src")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("arena-shuffle")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .getOrCreate()
)

RAW = "../../data/raw"
EVENTOS = "../../data/events"

spark

A célula abaixo já vem pronta: o mesmo código do vídeo 4.1 (feed, partidas, configuração herdada e relatório).

In [ ]:
# vídeo 4.1: o feed e as partidas
from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas

eventos = spark.read.parquet(EVENTOS).withColumnRenamed("MatchID", "match_id")

partidas = limpar_partidas(
    spark.read.option("header", "true").schema(MATCHES).csv(f"{RAW}/WorldCupMatches.csv")
)

# vídeo 4.1: a configuração herdada do job
# configuração herdada do job da operação
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

# vídeo 4.1: o relatório da operação
def relatorio_da_operacao():
    """Quantos lances de cada tipo aconteceram em cada Copa."""
    return (
        eventos.join(partidas, on="match_id")
        .groupBy("ano", "event_type")
        .count()
    )

## O que é um shuffle

A troca de dados entre partições, que aparece no plano como `Exchange`: cada task grava o que vai mandar, os dados
atravessam a rede e as tasks seguintes leem o que receberam.

## A agregação em duas fases

1. `partial_count`: cada partição conta os seus lances, antes da troca;
2. `Exchange`: o shuffle;
3. `count`: cada partição soma os parciais que recebeu.

| Tipo | Precisa de dado de outras partições? | Exemplos |
|---|---|---|
| **estreita** | não | `filter`, `select`, `withColumn` |
| **larga** | sim, exige shuffle | `groupBy`, `join`, `distinct`, `orderBy`, `dropDuplicates` |

## O número 200

`spark.sql.shuffle.partitions`: o número de partições depois de um shuffle. Padrão: **200**.

126 linhas de resultado em 200 partições: **200 tasks**, quase todas vazias. Medindo com 200 e com 8:

O 8 resolve **para este volume**; acertar na mão a cada rodada não escala (vídeo 4.3). De volta ao valor do job:

## Quantas linhas ficam em cada partição

`F.spark_partition_id()`: o número da partição de cada linha.

## repartition e coalesce

- **`repartition(n)`**: redistribui **com shuffle**; partições equilibradas; sobe ou desce.
- **`coalesce(n)`**: junta partições vizinhas, **sem shuffle**; só diminui; pode desequilibrar.

| Situação | Use |
|---|---|
| diminuir o número de arquivos ao gravar (aula 5) | `coalesce` |
| corrigir partições desequilibradas | `repartition` |

## Resultado da task 2

Os três `Exchange` são shuffles, e 200 partições era demais para o resultado. Próxima task: tirar a configuração
herdada.

## Para consultar depois

- **`repartition("coluna")`** redistribui pelo valor da coluna: todas as linhas de um mesmo valor vão para a mesma
  partição. Ele volta na aula 5, na hora de gravar.
- **`orderBy`** também é uma transformação larga: para ordenar o dado inteiro, cada faixa de valores vai para uma
  partição (`rangepartitioning` no plano).
- **`coalesce` no plano** não gera `Exchange`; `repartition` gera um.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [ ]:
spark.stop()